# NFL Draft Prediction: The Ultimate Pipeline
This notebook synthesizes the **Master Branch** (Logistic Regression Stacker), **V6** (Greedy Hill-Climbing), and **V7** (TabNet Deep Learning) architectures into a single pipeline optimized for Google Colab.

In [ ]:
!pip install optuna pytorch-tabnet catboost lightgbm xgboost scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import optuna
import warnings
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
POWER_5_SCHOOLS = {
    'Alabama', 'LSU', 'Georgia', 'Florida', 'Auburn', 'Arkansas', 'Texas A&M', 'Missouri', 
    'South Carolina', 'Ole Miss', 'Mississippi St.', 'Tennessee', 'Kentucky', 'Vanderbilt', 'Mississippi',
    'Ohio St.', 'Penn St.', 'Wisconsin', 'Michigan', 'Michigan St.', 'Iowa', 'Nebraska', 
    'Minnesota', 'Illinois', 'Northwestern', 'Purdue', 'Indiana', 'Rutgers', 'Maryland',
    'Clemson', 'Florida St.', 'Miami (FL)', 'Virginia Tech', 'North Carolina', 'Duke', 
    'Georgia Tech', 'Louisville', 'NC State', 'Pittsburgh', 'Boston College', 'Syracuse', 
    'Virginia', 'Wake Forest', 'Miami',
    'Oklahoma', 'Texas', 'West Virginia', 'TCU', 'Baylor', 'Oklahoma St.', 'Kansas St.', 
    'Iowa St.', 'Texas Tech', 'Kansas', 'BYU', 'Houston', 'UCF', 'Cincinnati',
    'USC', 'UCLA', 'Stanford', 'Oregon', 'Washington', 'Utah', 'Arizona St.', 'California', 
    'Washington St.', 'Oregon St.', 'Colorado', 'Arizona', 'Notre Dame'
}

def get_archetype(row):
    h, w = row['Height'], row['Weight']
    if pd.isnull(h) or pd.isnull(w):
        return 'Lightweight_Skills'
    if w >= 120:
        return 'Heavyweight_Lineman'
    elif w >= 95:
        if h >= 1.90:
            return 'Big_Hybrid_Edge_TE'
        else:
            return 'Dense_Power_LB_RB'
    else:
        if h >= 1.88:
            return 'Tall_Speed_WR_DB'
        else:
            return 'Lightweight_Skills'

def engineer_features(df):
    df = df.copy()
    df['Power_Factor'] = df['Weight'] * df['Vertical_Jump']
    df['Speed_to_Size_Ratio'] = df['Sprint_40yd'] / df['Weight']
    df['Total_Jump'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Agility_Score'] = df['Agility_3cone'] + df['Shuttle']
    df['BMI'] = (df['Weight'] / (df['Height'] ** 2)) * 703
    df['Momentum'] = df['Weight'] * (40.0 / df['Sprint_40yd'])
    df['Jump_Power_Index'] = df['Weight'] * df['Total_Jump']
    df['Agility_Speed_Ratio'] = df['Agility_Score'] / df['Sprint_40yd']
    df['School_Conference'] = df['School'].apply(lambda x: 'Power_5' if x in POWER_5_SCHOOLS else 'Other_Conference')
    df['Physical_Archetype'] = df.apply(get_archetype, axis=1)
    return df

In [ ]:
class PreprocessingPipeline:
    def __init__(self, cat_cols, num_cols_to_clip=None, impute_and_scale=False):
        self.cat_cols = cat_cols
        self.num_cols_to_clip = num_cols_to_clip
        self.impute_and_scale = impute_and_scale
        self.cat_dtypes = {}
        self.preprocessor = None

    def _get_num_cols(self, df):
        return [c for c in df.columns if c not in self.cat_cols]

    def fit_transform(self, X, y=None):
        X = X.copy()
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).fillna('missing')
                cat_type = pd.CategoricalDtype(categories=list(X[col].unique()) + ['missing'])
                X[col] = X[col].astype(cat_type)
                self.cat_dtypes[col] = cat_type
        
        num_cols = self._get_num_cols(X)
        if self.num_cols_to_clip:
            for col in self.num_cols_to_clip:
                if col in X.columns:
                    X[col] = X[col].clip(lower=X[col].quantile(0.01), upper=X[col].quantile(0.99))
        
        if self.impute_and_scale:
            num_transformer = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ])
            cat_transformer = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ])
            self.preprocessor = ColumnTransformer(transformers=[
                ('num', num_transformer, num_cols),
                ('cat', cat_transformer, self.cat_cols)
            ], remainder='passthrough')
            
            X_trans = self.preprocessor.fit_transform(X)
            return pd.DataFrame(X_trans, columns=self.preprocessor.get_feature_names_out(), index=X.index)
        
        return X

    def transform(self, X):
        X = X.copy()
        if not self.impute_and_scale:
            for col in self.cat_cols:
                if col in X.columns:
                    s = X[col].astype(str)
                    valid_cats = set(self.cat_dtypes[col].categories)
                    s = s.map(lambda x: x if x in valid_cats else 'missing')
                    X[col] = s.astype(self.cat_dtypes[col])
            
            if self.num_cols_to_clip:
                for col in self.num_cols_to_clip:
                    if col in X.columns:
                        X[col] = X[col].clip(lower=X[col].quantile(0.01), upper=X[col].quantile(0.99))
            return X
            
        if self.num_cols_to_clip:
            for col in self.num_cols_to_clip:
                if col in X.columns:
                    X[col] = X[col].clip(lower=X[col].quantile(0.01), upper=X[col].quantile(0.99))
                    
        X_trans = self.preprocessor.transform(X)
        return pd.DataFrame(X_trans, columns=self.preprocessor.get_feature_names_out(), index=X.index)

class TabNetPreprocessor:
    def __init__(self, cat_cols):
        self.cat_cols = cat_cols
        self.encoders = {}
        self.cat_dims = []
        self.cat_idxs = []
        self.num_imputer = SimpleImputer(strategy='median')
        self.num_scaler = StandardScaler()

    def fit_transform(self, X, y=None):
        X = X.copy()
        self.cat_idxs = [X.columns.get_loc(c) for c in self.cat_cols if c in X.columns]
        num_cols = [c for c in X.columns if c not in self.cat_cols]
        
        X[num_cols] = self.num_imputer.fit_transform(X[num_cols])
        X[num_cols] = self.num_scaler.fit_transform(X[num_cols])
        
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).fillna('missing')
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col])
                self.encoders[col] = le
                self.cat_dims.append(len(le.classes_) + 1)  # +1 for unknown in test
        return X

    def transform(self, X):
        X = X.copy()
        num_cols = [c for c in X.columns if c not in self.cat_cols]
        X[num_cols] = self.num_imputer.transform(X[num_cols])
        X[num_cols] = self.num_scaler.transform(X[num_cols])
        
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).fillna('missing')
                le = self.encoders[col]
                unknown_class = len(le.classes_)
                X[col] = X[col].apply(lambda x: np.where(le.classes_ == x)[0][0] if x in le.classes_ else unknown_class)
        return X

In [ ]:
def train_and_validate(X, y, model_factory, n_folds=5, cat_cols=None, num_cols_to_clip=None):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=False)
    scores = []
    oof_probs = np.zeros(len(X))
    fitted_models = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_train_raw, X_val_raw = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = model_factory()
        model_name = type(model).__name__.lower()
        
        if "tabnet" in model_name:
            pipeline = TabNetPreprocessor(cat_cols)
            X_train = pipeline.fit_transform(X_train_raw).values
            X_val = pipeline.transform(X_val_raw).values
            model.cat_idxs = pipeline.cat_idxs
            model.cat_dims = pipeline.cat_dims
            model.fit(
                X_train, y_train.values,
                eval_set=[(X_val, y_val.values)],
                eval_metric=['auc'],
                max_epochs=100,
                patience=20,
                batch_size=256,
                virtual_batch_size=128
            )
            probs = model.predict_proba(X_val)[:, 1]
        else:
            is_linear = "logisticregression" in model_name
            pipeline = PreprocessingPipeline(cat_cols, num_cols_to_clip=num_cols_to_clip, impute_and_scale=is_linear)
            X_train = pipeline.fit_transform(X_train_raw, y_train)
            X_val = pipeline.transform(X_val_raw)
            
            fit_kwargs = {}
            if "xgb" in model_name:
                fit_kwargs['eval_set'] = [(X_val, y_val)]
                fit_kwargs['verbose'] = False
            elif "lgbm" in model_name:
                fit_kwargs['eval_set'] = [(X_val, y_val)]
            elif "catboost" in model_name:
                fit_kwargs['eval_set'] = (X_val, y_val)
                fit_kwargs['verbose'] = False
                
            model.fit(X_train, y_train, **fit_kwargs)
            probs = model.predict_proba(X_val)[:, 1]
            
        oof_probs[val_idx] = probs
        scores.append(roc_auc_score(y_val, probs))
        fitted_models.append((pipeline, model))
        
    return np.mean(scores), oof_probs, fitted_models

def objective_xgb(trial, X, y, cat_cols, num_cols_to_clip):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    factory = lambda: XGBClassifier(**params, use_label_encoder=False, eval_metric='logloss', tree_method='hist', enable_categorical=True, early_stopping_rounds=50)
    score, _, _ = train_and_validate(X, y, factory, n_folds=5, cat_cols=cat_cols, num_cols_to_clip=num_cols_to_clip)
    return score

def objective_lgbm(trial, X, y, cat_cols, num_cols_to_clip):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    factory = lambda: LGBMClassifier(**params, verbose=-1, early_stopping_rounds=50)
    score, _, _ = train_and_validate(X, y, factory, n_folds=5, cat_cols=cat_cols, num_cols_to_clip=num_cols_to_clip)
    return score

def objective_cat(trial, X, y, cat_cols, num_cols_to_clip):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True)
    }
    factory = lambda: CatBoostClassifier(**params, verbose=0, eval_metric='AUC', cat_features=cat_cols, early_stopping_rounds=50)
    score, _, _ = train_and_validate(X, y, factory, n_folds=5, cat_cols=cat_cols, num_cols_to_clip=num_cols_to_clip)
    return score

def objective_tabnet(trial, X, y, cat_cols, num_cols_to_clip):
    n_d = trial.suggest_int('n_d', 8, 64)
    n_a = n_d
    n_steps = trial.suggest_int('n_steps', 3, 10)
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    n_independent = trial.suggest_int('n_independent', 1, 5)
    n_shared = trial.suggest_int('n_shared', 1, 5)
    momentum = trial.suggest_float('momentum', 0.01, 0.4)
    
    factory = lambda: TabNetClassifier(
        n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
        n_independent=n_independent, n_shared=n_shared, momentum=momentum,
        verbose=0
    )
    score, _, _ = train_and_validate(X, y, factory, n_folds=5, cat_cols=cat_cols, num_cols_to_clip=num_cols_to_clip)
    return score

In [ ]:
# Load Data (Please ensure this path is correct for your Google Drive)
train = pd.read_csv('/content/drive/MyDrive/GCI_Global/competition/input/train.csv')
test = pd.read_csv('/content/drive/MyDrive/GCI_Global/competition/input/test.csv')

train = engineer_features(train)
test = engineer_features(test)

X = train.drop(['Drafted', 'Id'], axis=1)
y = train['Drafted']
X_test_raw = test.drop(['Id'], axis=1)

cat_cols = ['School', 'Position', 'Player_Type', 'Position_Type', 'Year', 'School_Conference', 'Physical_Archetype']

num_cols_to_clip = [
    'Age', 'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump', 
    'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle',
    'BMI', 'Speed_Score', 'Power_Factor', 'Speed_to_Size_Ratio', 
    'Total_Jump', 'Agility_Score', 'Explosiveness_Index',
    'Bench_to_Weight_Ratio', 'Speed_to_Weight_Efficiency', 
    'Explosiveness_to_Weight_Efficiency', 'Momentum', 'Jump_Power_Index', 'Agility_Speed_Ratio'
]

In [ ]:
# Set n_trials (e.g. 100 for final run, 20 for testing)
N_TRIALS = 100

print("Optimizing XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(lambda t: objective_xgb(t, X, y, cat_cols, num_cols_to_clip), n_trials=N_TRIALS)

print("Optimizing LightGBM...")
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lambda t: objective_lgbm(t, X, y, cat_cols, num_cols_to_clip), n_trials=N_TRIALS)

print("Optimizing CatBoost...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(lambda t: objective_cat(t, X, y, cat_cols, num_cols_to_clip), n_trials=N_TRIALS)

print("Optimizing TabNet...")
study_tabnet = optuna.create_study(direction='maximize')
study_tabnet.optimize(lambda t: objective_tabnet(t, X, y, cat_cols, num_cols_to_clip), n_trials=N_TRIALS)

In [ ]:
models = {
    'xgb': lambda: XGBClassifier(**study_xgb.best_params, use_label_encoder=False, eval_metric='logloss', tree_method='hist', enable_categorical=True, early_stopping_rounds=50),
    'lgbm': lambda: LGBMClassifier(**study_lgbm.best_params, verbose=-1, early_stopping_rounds=50),
    'cat': lambda: CatBoostClassifier(**study_cat.best_params, verbose=0, eval_metric='AUC', cat_features=cat_cols, early_stopping_rounds=50),
    'tabnet': lambda: TabNetClassifier(**study_tabnet.best_params, verbose=0)
}

results = {}
test_probs = {}

for name, factory_fn in models.items():
    print(f"Training 5-Fold CV for {name.upper()}...")
    score, oof, fitted_models = train_and_validate(X, y, factory_fn, n_folds=5, cat_cols=cat_cols, num_cols_to_clip=num_cols_to_clip)
    results[name] = {'score': score, 'oof': oof}
    print(f"{name.upper()} CV ROC AUC: {score:.5f}")
    
    # Predict on test set
    model_test_probs = np.zeros(len(test))
    for pipeline, model in fitted_models:
        if name == 'tabnet':
            X_test_processed = pipeline.transform(X_test_raw).values
            model_test_probs += model.predict_proba(X_test_processed)[:, 1] / len(fitted_models)
        else:
            X_test_processed = pipeline.transform(X_test_raw)
            model_test_probs += model.predict_proba(X_test_processed)[:, 1] / len(fitted_models)
            
    test_probs[name] = model_test_probs

In [ ]:
# Master Branch Winner (Logistic Regression Stacker without TabNet)
oof_matrix_master = np.column_stack([results['xgb']['oof'], results['lgbm']['oof'], results['cat']['oof']])
test_matrix_master = np.column_stack([test_probs['xgb'], test_probs['lgbm'], test_probs['cat']])

meta_lr = LogisticRegression(C=1.0, max_iter=1000)
meta_lr.fit(oof_matrix_master, y)
master_score = roc_auc_score(y, meta_lr.predict_proba(oof_matrix_master)[:, 1])
print(f"Master Branch (Logistic Regression) CV ROC AUC: {master_score:.5f}")

# V6 Strategy (Greedy Hill-Climbing Stacker without TabNet)
iterations = 100
ensemble_oof = np.zeros(len(y))
ensemble_test = np.zeros(len(test))
model_weights = np.zeros(oof_matrix_master.shape[1])

for i in range(iterations):
    best_auc = 0
    best_idx = 0
    best_oof = None
    for m_idx in range(oof_matrix_master.shape[1]):
        temp_oof = (ensemble_oof * i + oof_matrix_master[:, m_idx]) / (i + 1)
        temp_auc = roc_auc_score(y, temp_oof)
        if temp_auc > best_auc:
            best_auc, best_idx, best_oof = temp_auc, m_idx, temp_oof
    ensemble_oof = best_oof
    ensemble_test = (ensemble_test * i + test_matrix_master[:, best_idx]) / (i + 1)
    model_weights[best_idx] += 1

v6_score = roc_auc_score(y, ensemble_oof)
print(f"V6 Branch (Hill Climbing) CV ROC AUC: {v6_score:.5f}")

# V7 Strategy (TabNet Ensembling with Logistic Regression)
oof_matrix_v7 = np.column_stack([oof_matrix_master, results['tabnet']['oof']])
test_matrix_v7 = np.column_stack([test_matrix_master, test_probs['tabnet']])

meta_v7 = LogisticRegression(C=1.0, max_iter=1000)
meta_v7.fit(oof_matrix_v7, y)
v7_score = roc_auc_score(y, meta_v7.predict_proba(oof_matrix_v7)[:, 1])
print(f"V7 Branch (TabNet + Trees Logistic Regression) CV ROC AUC: {v7_score:.5f}")

# Save the absolute best one to submission.csv (Example: V7)
submission = pd.DataFrame({'Id': test['Id'], 'Drafted': meta_v7.predict_proba(test_matrix_v7)[:, 1]})
submission.to_csv('submission.csv', index=False)
print("Saved best predictions to submission.csv")